# Ensemble Model Exploration
## Decision Tree vs Random Forest vs Gradient Boosting

**Goal:** preprocess a standard tabular dataset, train three models on the same split, compare evaluation metrics, and save a Markdown report.

This notebook uses scikit-learn's built-in Breast Cancer Wisconsin dataset, so no separate dataset file is required. This is a machine-learning educational example, not a medical diagnostic tool.


In [ ]:
# 1. Import libraries
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
ROOT = Path.cwd()
# If notebook was launched from another folder, find project root.
if not (ROOT / "reports").exists() and (ROOT.parent / "reports").exists():
    ROOT = ROOT.parent
(ROOT / "reports").mkdir(exist_ok=True)
print("Project root:", ROOT.resolve())


In [ ]:
# 2. Load dataset and perform initial inspection
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()

print("Dataset shape:", df.shape)
display(df.head())
print("\\nData types:")
display(df.dtypes.value_counts().rename_axis("dtype").to_frame("count"))
print("\\nMissing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\\nTarget counts (0 = malignant, 1 = benign):")
display(df["target"].value_counts().sort_index().rename(index={0:"malignant", 1:"benign"}))


In [ ]:
# 3. Separate features/target and split into training/testing sets
X = df.drop(columns=["target"])
y = df["target"]

# This built-in dataset has numeric features and no missing values.
# If missing values exist, the median imputer handles them safely.
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


In [ ]:
# 4. Train three models on the same training data
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=4),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight="balanced"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
    })

results_df = pd.DataFrame(results).sort_values("F1-Score", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.3f}" for c in ["Accuracy","Precision","Recall","F1-Score"]}))


In [ ]:
# 5. Visualize model comparison
ax = results_df.set_index("Model")[["Accuracy", "F1-Score"]].plot(kind="bar", ylim=(0, 1), figsize=(9, 5))
ax.set_title("Decision Tree vs Ensemble Models")
ax.set_ylabel("Score")
ax.set_xlabel("Model")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# 6. Confusion matrices and classification reports
for name, y_pred in predictions.items():
    print(f"\\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=dataset.target_names, zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    display(pd.DataFrame(cm, index=[f"Actual {x}" for x in dataset.target_names],
                         columns=[f"Predicted {x}" for x in dataset.target_names]))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=dataset.target_names).plot()
    plt.title(f"Confusion Matrix: {name}")
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Save results to reports/performance_comparison.md
best = results_df.iloc[0]
lines = [
    "# Performance Comparison Report",
    "",
    "## Objective",
    "Compare a baseline Decision Tree with Random Forest and Gradient Boosting classifiers.",
    "",
    "## Dataset and preprocessing",
    f"- Dataset: {dataset.DESCR.splitlines()[0]}",
    f"- Rows: {len(df)}; features: {X.shape[1]}",
    f"- Missing values in source data: {int(df.isna().sum().sum())}",
    "- Stratified 80/20 train/test split; median imputation applied to numeric features.",
    "",
    "## Results",
    results_df.to_markdown(index=False, floatfmt=".3f"),
    "",
    f"Best F1-score in this run: **{best['Model']}** (F1 = {best['F1-Score']:.3f}).",
    "",
    "## Discussion",
    "- Decision Tree is relatively easy to interpret but may overfit.",
    "- Random Forest averages many trees and often improves robustness.",
    "- Gradient Boosting builds trees sequentially and can capture complex patterns.",
    "- Compare F1-score alongside accuracy so both false positives and false negatives are considered.",
    "",
    "## Limitations",
    "A single test split is only one estimate of generalization. This dataset is for education only and must not be used for medical diagnosis.",
]
report_path = ROOT / "reports" / "performance_comparison.md"
report_path.write_text("\\n".join(lines), encoding="utf-8")
print("Saved report to:", report_path.resolve())
print("\\nReport preview:\\n")
print(report_path.read_text(encoding="utf-8"))
